In [14]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import os

In [17]:
# Definimos la ruta de tu archivo local
CSV_PATH = "outputs/processed/sevilla_pixel_data.csv"

# Cargamos el DataFrame
df = pd.read_csv(CSV_PATH)

print(f"✅ Dataset cargado correctamente: {CSV_PATH}")
print(f"Total de píxeles (registros): {len(df)}")
print(f"Variables disponibles: {list(df.columns)}")

# Mostramos las primeras filas para verificar
df.head()

✅ Dataset cargado correctamente: outputs/processed/sevilla_pixel_data.csv
Total de píxeles (registros): 484356
Variables disponibles: ['Longitude', 'Latitude', 'NDVI', 'NDBI', 'Albedo', 'D2W_meters', 'LST_Target', 'D2R_HighCapacity_m', 'D2R_Urban_m', 'Tree_Density_50m', 'Building_Density_100m', 'Avg_Building_Height_100m']


,Longitude,Latitude,NDVI,NDBI,Albedo,D2W_meters,LST_Target,D2R_HighCapacity_m,D2R_Urban_m,Tree_Density_50m,Building_Density_100m,Avg_Building_Height_100m
0,-6.029941,37.449956,0.505166,-0.178238,0.157998,20.000000,40.960874,2743.999249,2776.807410,1,0,0.0
1,-6.029762,37.449956,0.728687,-0.395285,0.181745,20.000000,40.714777,2743.879563,2760.939843,1,0,0.0
2,-6.029582,37.449956,0.763079,-0.424442,0.149877,20.000000,40.714777,2743.851804,2745.072446,1,0,0.0
3,-6.029402,37.449956,0.775685,-0.418584,0.147389,22.360680,40.379811,2743.915975,2729.205220,1,0,0.0
4,-6.029223,37.449956,0.835151,-0.530323,0.201774,28.284271,40.379811,2744.072070,2713.338171,1,0,0.0


In [18]:

# 2. Convertimos a GeoDataFrame
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.Longitude, df.Latitude))

# 3. Cargamos los barrios de Sevilla (GeoJSON)
barrios = gpd.read_file('outputs/raw/barrios_sevilla.geojson')

# 4. Asignamos cada punto a un barrio (Spatial Join)
# Esto une los puntos con los polígonos de los barrios automáticamente
gdf_joined = gpd.sjoin(gdf, barrios, how="inner", predicate="within")

columnas_media = [
    'NDVI', 'NDBI', 'Albedo', 'D2W_meters', 'LST_Target',
    'D2R_HighCapacity_m', 'D2R_Urban_m', 'Tree_Density_50m',
    'Building_Density_100m', 'Avg_Building_Height_100m'
]

# 5. Calculamos la media de temperatura por barrio
resumen_barrios_df = gdf_joined.groupby('name')[columnas_media].mean().reset_index()

# Hacemos un 'merge' con el GeoDataFrame original de barrios para recuperar sus políGONOS
resumen_barrios_gdf = resumen_barrios_df.merge(barrios[['name', 'geometry']], on='name')

# Convertimos a GeoDataFrame definitivo
resumen_barrios_gdf = gpd.GeoDataFrame(resumen_barrios_gdf, geometry='geometry')

# Guardamos
resumen_barrios_gdf.to_file("outputs/processed/mapa_barrios_temperatura.geojson", driver='GeoJSON')
resumen_barrios_gdf.to_file("web/frontend-sevilla/src/assets/mapa_barrios_temperatura.geojson", driver='GeoJSON')


In [16]:
# 7. NUEVO: Crear un CSV por cada barrio
print("💾 Generando archivos CSV detallados por barrio...")

# Definimos las rutas de destino
rutas_destino = ["outputs/processed/barrios", "web/frontend-sevilla/src/assets/barrios"]

for nombre_barrio, grupo in gdf_joined.groupby('name'):
    # Limpiamos el nombre para que sea un nombre de archivo válido (quitamos espacios y pasamos a minúsculas)
    nombre_archivo = f"detail_{nombre_barrio.replace(' ', '_').lower()}.csv"
    
    for ruta in rutas_destino:
        # Aseguramos que el directorio exista
        os.makedirs(ruta, exist_ok=True)
        
        # Guardamos el CSV filtrado para este barrio
        # (Si solo quieres ciertas columnas, puedes poner .to_csv(ruta + nombre_archivo, columns=['col1', 'col2', ...]))
        grupo.to_csv(os.path.join(ruta, nombre_archivo), index=False, encoding='utf-8')

print("✅ Todos los archivos CSV han sido generados exitosamente.")

💾 Generando archivos CSV detallados por barrio...
✅ Todos los archivos CSV han sido generados exitosamente.
